# 19. Ablation studies: isolating the LLM contribution

Three ablation experiments that answer the question: *"How much of LG-CoTrain's performance comes from GPT-4o pseudo-labels?"*

All three variants run with the same default hyperparameters (no Optuna tuning; for tuned versions see NB27 for self-trained and NB29 for vanilla-cotrain). Together with the LG-CoTrain baseline, they isolate the contribution of the LLM teacher from the pipeline mechanics.

| # | Name | What changes vs LG-CoTrain baseline |
|---|---|---|
| 1 | **self-trained** | Replace GPT-4o pseudo-labels with self-trained BERTweet teacher predictions (all) |
| 2 | **self-trained-top-p** | Same as self-trained, but keep only top-p (50) most confident predictions per class |
| 3 | **vanilla-cotrain** | No external teacher. Two BERTweet models teach each other (Blum & Mitchell 1998) |

Baseline reference tree: `results/bertweet/gpt-4o/defaults/wge7/` (BERTweet + defaults + GPT-4o, 120 cells, produced by NB14).

## Results paths

```
results/bertweet/ablation/self-trained/baseline/         <- this notebook
results/bertweet/ablation/self-trained-top-p/baseline/   <- this notebook
results/bertweet/ablation/vanilla-cotrain/baseline/      <- this notebook
```


## Setup

In [ ]:
import sys
import subprocess
import time
import json
import statistics
from collections import defaultdict
from pathlib import Path

def _find_repo_root(marker: str = "lg_cotrain") -> Path:
    for candidate in [Path().resolve()] + list(Path().resolve().parents):
        if (candidate / marker).is_dir():
            return candidate
    raise RuntimeError(f"Cannot find repo root: no ancestor directory contains '{marker}/'")

repo_root = _find_repo_root()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"Repo root: {repo_root}")
print(f"Python:    {sys.version.split()[0]}")

In [ ]:
import torch
print(f"torch version:     {torch.__version__}")
print(f"CUDA available:    {torch.cuda.is_available()}")
N_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 1
print(f"CUDA device count: {N_GPUS}")
for i in range(N_GPUS):
    props = torch.cuda.get_device_properties(i)
    free_gb, total_gb = (x / 1024**3 for x in torch.cuda.mem_get_info(i))
    print(f"  cuda:{i}  {props.name}  total={total_gb:.1f} GB  free={free_gb:.1f} GB")

In [ ]:
# Configuration
EVENTS = [
    "kaikoura_earthquake_2016",
    "canada_wildfires_2016",
    "cyclone_idai_2019",
    "hurricane_florence_2018",
    "hurricane_maria_2017",
    "california_wildfires_2018",
    "hurricane_dorian_2019",
    "kerala_floods_2018",
    "hurricane_harvey_2017",
    "hurricane_irma_2017",
]
BUDGETS = [5, 10, 25, 50]
SEEDS = [1, 2, 3]
MODEL_NAME = "vinai/bertweet-base"
DATA_ROOT = repo_root / "data"
EXPECTED_CELLS = len(EVENTS) * len(BUDGETS) * len(SEEDS)  # 120

# Output paths
ABLATION_ROOT = repo_root / "results" / "bertweet" / "ablation"
PATH_SELF_TRAINED = ABLATION_ROOT / "self-trained" / "baseline"
PATH_TOP_P = ABLATION_ROOT / "self-trained-top-p" / "baseline"
PATH_VANILLA = ABLATION_ROOT / "vanilla-cotrain" / "baseline"

# Baseline for comparison
BASELINE_PATH = repo_root / "results" / "bertweet" / "gpt-4o" / "defaults" / "wge7"

print(f"Events:            {len(EVENTS)}")
print(f"Budgets:           {BUDGETS}")
print(f"Seeds:             {SEEDS}")
print(f"Expected cells:    {EXPECTED_CELLS}")
print(f"Model:             {MODEL_NAME}")
print(f"Num GPUs:          {N_GPUS}")
print()

# Count existing results
for label, path in [
    ("Baseline (GPT-4o)", BASELINE_PATH),
    ("self-trained", PATH_SELF_TRAINED),
    ("self-trained-top-p", PATH_TOP_P),
    ("vanilla-cotrain", PATH_VANILLA),
]:
    n = len(list(path.glob("*/*/metrics.json"))) if path.exists() else 0
    print(f"  {label:<25s} {n:>3d}/{EXPECTED_CELLS}  [{path.relative_to(repo_root)}]")

## Ablation 2: self-trained-top-p

Filter the existing self-trained teacher CSVs to keep only the top-50 most confident predictions per class, then re-run LG-CoTrain with the filtered pseudo-labels.

**Step 1**: Generate filtered CSVs (seconds, no GPU).
**Step 2**: Run LG-CoTrain with `--pseudo-label-source self-trained-top-p` (120 cells).

In [ ]:
# Ablation 2 Step 1: filter self-trained pseudo-labels to top-50 per class
SAMPLES_PER_CLASS = 50

cmd = [
    sys.executable, "-m", "lg_cotrain.filter_pseudo_labels",
    "--samples-per-class", str(SAMPLES_PER_CLASS),
    "--data-root", str(DATA_ROOT),
]
print("Filtering pseudo-labels:")
print("  " + " ".join(cmd))
print()

proc = subprocess.run(cmd, cwd=str(repo_root))
if proc.returncode != 0:
    raise RuntimeError(f"Filter script failed with exit code {proc.returncode}")

# Verify
filtered_dir = DATA_ROOT / "pseudo-labelled" / "self-trained-top-p"
n_filtered = len(list(filtered_dir.glob("*/labeled_*_pseudo.csv")))
print(f"\nFiltered CSVs on disk: {n_filtered} / {EXPECTED_CELLS}")

In [ ]:
# Ablation 2 Step 2: run LG-CoTrain with filtered pseudo-labels
PATH_TOP_P.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, "-m", "lg_cotrain.run_experiment",
    "--events", *EVENTS,
    "--budgets", *(str(b) for b in BUDGETS),
    "--seed-sets", *(str(s) for s in SEEDS),
    "--num-gpus", str(N_GPUS),
    "--model-name", MODEL_NAME,
    "--pseudo-label-source", "self-trained-top-p",
    "--data-root", str(DATA_ROOT),
    "--output-folder", str(PATH_TOP_P),
]
print("Running self-trained-top-p ablation:")
print("  " + " ".join(cmd))
print()
print(f"Expected: {EXPECTED_CELLS} LG-CoTrain runs (skipping any already done)")
print("=" * 70)

start = time.time()
proc = subprocess.run(cmd, cwd=str(repo_root))
elapsed = time.time() - start

print("=" * 70)
print(f"Wall clock: {elapsed:.0f}s = {elapsed/60:.1f}min = {elapsed/3600:.2f}h")
if proc.returncode != 0:
    print(f"WARNING: exit code {proc.returncode}. Some cells may have failed.")

n_done = len(list(PATH_TOP_P.glob("*/*/metrics.json")))
print(f"\nmetrics.json files: {n_done} / {EXPECTED_CELLS}")
if n_done == EXPECTED_CELLS:
    print("ABLATION 2 COMPLETE")
else:
    print(f"MISSING: {EXPECTED_CELLS - n_done} cells. Re-run this cell to retry.")

## Ablation 3: vanilla-cotrain

Classic Blum & Mitchell co-training. No external teacher, no lambda weighting. The two BERTweet models start with D_l1/D_l2, then iteratively:
1. Train both models on their current labeled sets (5 epochs per iteration)
2. Both predict on the remaining unlabeled pool
3. Select top-5 most confident per class from each model
4. Exchange: model A's selections go to model B's training set, and vice versa
5. Remove selected samples from the unlabeled pool

After 10 iterations, fine-tune both models with early stopping and produce ensemble predictions.

**Note:** This run used `--samples-per-class 5 --num-iterations 10` (the original defaults at the time). The defaults have since been updated to `samples_per_class=1, num_iterations=30` to match Blum & Mitchell (1998).

In [ ]:
# Ablation 3: run vanilla co-training
# Explicit --samples-per-class 5 --num-iterations 10 to preserve original behavior
# (defaults have since changed to 1/30 to match Blum & Mitchell 1998)
PATH_VANILLA.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, "-m", "vanilla_cotrain.run_experiment",
    "--events", *EVENTS,
    "--budgets", *(str(b) for b in BUDGETS),
    "--seed-sets", *(str(s) for s in SEEDS),
    "--num-gpus", str(N_GPUS),
    "--model-name", MODEL_NAME,
    "--samples-per-class", "5",
    "--num-iterations", "10",
    "--data-root", str(DATA_ROOT),
    "--output-folder", str(PATH_VANILLA),
]
print("Running vanilla co-training ablation:")
print("  " + " ".join(cmd))
print()
print(f"Expected: {EXPECTED_CELLS} vanilla co-training runs (skipping any already done)")
print("=" * 70)

start = time.time()
proc = subprocess.run(cmd, cwd=str(repo_root))
elapsed = time.time() - start

print("=" * 70)
print(f"Wall clock: {elapsed:.0f}s = {elapsed/60:.1f}min = {elapsed/3600:.2f}h")
if proc.returncode != 0:
    print(f"WARNING: exit code {proc.returncode}. Some cells may have failed.")

n_done = len(list(PATH_VANILLA.glob("*/*/metrics.json")))
print(f"\nmetrics.json files: {n_done} / {EXPECTED_CELLS}")
if n_done == EXPECTED_CELLS:
    print("ABLATION 3 COMPLETE")
else:
    print(f"MISSING: {EXPECTED_CELLS - n_done} cells. Re-run this cell to retry.")

## Aggregation: compare all four variants

Loads metrics.json from all four trees (baseline + 3 ablations) and computes per-budget mean test macro-F1 with paired deltas.

In [ ]:
# Load all four trees
def load_tree(root):
    out = {}
    if not root.exists():
        return out
    for f in sorted(root.glob("*/*/metrics.json")):
        try:
            m = json.loads(f.read_text(encoding="utf-8"))
        except Exception:
            continue
        ev = f.parent.parent.name
        b, _, s = f.parent.name.partition("_")
        try:
            out[(ev, int(b), int(s.replace("set", "")))] = m
        except ValueError:
            continue
    return out

trees = {
    "Baseline (GPT-4o)":      load_tree(BASELINE_PATH),
    "self-trained":            load_tree(PATH_SELF_TRAINED),
    "self-trained-top-p":      load_tree(PATH_TOP_P),
    "vanilla-cotrain":         load_tree(PATH_VANILLA),
}

print("Loaded cells per tree:")
for label, tree in trees.items():
    print(f"  {label:<25s} {len(tree):>3d} / {EXPECTED_CELLS}")

baseline = trees["Baseline (GPT-4o)"]
if not baseline:
    raise RuntimeError("Baseline tree is empty. Check BASELINE_PATH.")

# Per-budget comparison
print()
print("=" * 110)
print("PER-BUDGET MEAN test_macro_f1 (mean +/- std across 30 event x seed cells)")
print("=" * 110)
header = f'{"budget":>6s}'
for label in trees:
    header += f"  {label:>25s}"
print(header)
print("-" * 110)

for b in BUDGETS:
    row = f"{b:>6d}"
    for label, tree in trees.items():
        vals = [tree[k]["test_macro_f1"] for k in tree if k[1] == b]
        if vals:
            m = sum(vals) / len(vals)
            row += f"  {m:>25.4f}"
        else:
            row += f"  {'no data':>25s}"
    print(row)

print("-" * 110)
row = f"{'all':>6s}"
for label, tree in trees.items():
    vals = [m["test_macro_f1"] for m in tree.values()]
    if vals:
        row += f"  {sum(vals)/len(vals):>25.4f}"
    else:
        row += f"  {'no data':>25s}"
print(row)

# Deltas vs baseline
print()
print("=" * 90)
print("DELTA vs baseline (negative = GPT-4o wins)")
print("=" * 90)
abl_labels = list(trees.keys())[1:]
header = f'{"budget":>6s}'
for label in abl_labels:
    header += f"  {label:>25s}"
print(header)
print("-" * 90)

for b in BUDGETS:
    row = f"{b:>6d}"
    for label in abl_labels:
        tree = trees[label]
        paired = [(baseline[k]["test_macro_f1"], tree[k]["test_macro_f1"])
                  for k in sorted(set(baseline) & set(tree)) if k[1] == b]
        if paired:
            deltas = [a - bl for bl, a in paired]
            row += f"  {sum(deltas)/len(deltas):>+25.4f}"
        else:
            row += f"  {'no data':>25s}"
    print(row)

## Per-event delta table

For each ablation, the per-event mean delta vs baseline across all 12 (budget x seed) cells per event.

In [ ]:
import math

for abl_label in list(trees.keys())[1:]:
    abl = trees[abl_label]
    if not abl:
        print(f"\n{abl_label}: no results loaded. Skip.\n")
        continue
    paired = sorted(set(baseline) & set(abl))
    if not paired:
        print(f"\n{abl_label}: no paired cells. Skip.\n")
        continue

    print("=" * 100)
    print(f"DELTA: ({abl_label}) - (Baseline)")
    print(f"Paired cells: {len(paired)}")
    print("=" * 100)
    print(f'{"event":<32s}' + "".join(f"  {b:>+8d}" for b in BUDGETS) + f'  {"mean":>8s}')
    print("-" * 100)

    for ev in EVENTS:
        cells = []
        for b in BUDGETS:
            ds = [abl[(ev, b, s)]["test_macro_f1"] - baseline[(ev, b, s)]["test_macro_f1"]
                  for s in SEEDS if (ev, b, s) in abl and (ev, b, s) in baseline]
            cells.append(sum(ds) / len(ds) if ds else None)
        present = [v for v in cells if v is not None]
        row_parts = "".join(
            f"  {v:>+8.4f}" if v is not None else f'  {"-":>8s}'
            for v in cells
        )
        rm = sum(present) / len(present) if present else None
        marker = ""
        if rm is not None:
            if rm > 0.005: marker = "  ^"
            elif rm < -0.005: marker = "  v"
        mean_str = f"{rm:>+8.4f}{marker}" if rm is not None else f'{"-":>8s}'
        print(f"{ev:<32s}{row_parts}  {mean_str}")
    print()

## Done

After all three ablations complete, re-run the aggregation cells above to see the full comparison. The expected ordering at each budget is:

```
Baseline (GPT-4o) > vanilla-cotrain > self-trained-top-p > self-trained
```

The per-budget deltas and per-event breakdown can be used directly in the paper's ablation section.